# vectorless rag

In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

os.environ["PAGEINDEX_API_KEY"] = os.getenv("PAGEINDEX_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from pageindex import PageIndexClient
from langchain_groq import ChatGroq

client = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))

llm = ChatGroq(
    model_name="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.5,
)





/Users/vikash/timescode/test/ragVectorless/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# upload pdf to build hierarchical tree index 

In [2]:
pdf_file = "../data/pdf/Jest-Unit-Test-Coverage-Report.pdf"

print(f'Uploading {pdf_file} to PageIndex...')

result = client.submit_document(pdf_file)
doc_id = result['doc_id']
print(result , doc_id)

Uploading ../data/pdf/Jest-Unit-Test-Coverage-Report.pdf to PageIndex...
{'doc_id': 'pi-cmrugg3u701yb01qujalt94lj'} pi-cmrugg3u701yb01qujalt94lj


## save id for use 

In [7]:
## build tree index 

print(f'Building tree index for {doc_id}...')
print('this run once per document ')

while True:
    status_result = client.get_document(doc_id)
    status = status_result['status']
    if status == 'completed':
        print(f'Tree index for {doc_id} completed')
        break
    elif status == 'failed':
        print(f'Tree index for {doc_id} failed')
        break
    else:
        print(f'Tree index for {doc_id} is {status}')
        time.sleep(5)

   


Building tree index for pi-cmrugg3u701yb01qujalt94lj...
this run once per document 
Tree index for pi-cmrugg3u701yb01qujalt94lj completed


In [8]:
## fetch the full tree
tree = client.get_tree(doc_id, node_summary=True)
page_index = tree.get('result', [])

print(f'Page Index for {doc_id} : {page_index}')

print("RAW TREE ")
print(tree)

def print_tree(nodes, indent=0):
    """Recursively print the tree (expects a list of node dicts)."""
    for node in nodes:
        pad = "  " * indent
        print(f"{pad}Node: {node['node_id']}")
        print(f"{pad}Title: {node.get('title', '')}")
        print(f"{pad}Summary: {node.get('summary') or node.get('prefix_summary', '')}")
        children = node.get('nodes') or []
        if children:
            print(f"{pad}Children ({len(children)}):")
            print_tree(children, indent + 1)
        print(f"{pad}{'-' * 50}")


# Pass the node list (page_index), not the raw API response dict (tree)
print_tree(page_index)


Page Index for pi-cmrugg3u701yb01qujalt94lj : [{'title': 'Jest Unit Test Coverage Report', 'node_id': '0000', 'page_index': 1, 'prefix_summary': '# Jest Unit Test Coverage Report\n\ntnn-english-next | Times Now News (English)\n\nReport date: 2026-07-14 | Jest coverage + __tests__ inventory\n', 'text': '# Jest Unit Test Coverage Report\n\ntnn-english-next | Times Now News (English)\n\nReport date: 2026-07-14 | Jest coverage + __tests__ inventory\n', 'nodes': [{'title': '1. Executive Summary', 'node_id': '0001', 'page_index': 1, 'summary': 'This report provides an overview of the TNN English Next.js Jest unit test suite, detailing its 324 test files, 3,277 static test cases, and 203 URL rewrite rules. It presents comprehensive coverage metrics (89.3% line coverage) and outlines the technical testing environment, which utilizes jest-environment-jsdom, Testing Library, and specific mocks for Next.js components.', 'text': '## 1. Executive Summary\n\nThis report inventories the Jest unit tes

In [9]:
### LLM tree search 
## vector rag retrival 


def llm_tree_search(query: str, tree: list, model):
    """core page index rag search
    query : str
    tree : list
    model : ChatGroq instance or model name str

    return : LLM response with thinking and node list
    """
    def compress(tree: list):
        out = []
        for node in tree:
            entry = {
                'node_id': node['node_id'],
                'title': node['title'],
                # Root nodes use prefix_summary; children use summary
                'summary': node.get('summary') or node.get('prefix_summary', ''),
                'page': node['page_index'],
            }
            if node.get('nodes'):
                entry['children'] = compress(node['nodes'])
            out.append(entry)
        return out

    compressed_tree = compress(tree)
    print(f"Compressed Tree: {compressed_tree}")

    prompt = f"""
    you are given query and a tree structure
    you need to answer the query based on the tree structure and node id that are relevant to the query
    query : {query}
    tree : {compressed_tree}
    reply only in this format :
    {{
        "thinking": "<thinking>",
        "node_id": "<node_id>",
        "title": "<title>",
        "summary": "<summary>",
        "page": "<page>"
    }}
    """
    llm_chain = model if hasattr(model, 'invoke') else ChatGroq(model=model, temperature=0.5)
    response = llm_chain.invoke(prompt)
    return response


In [10]:
query = "what is unit test case cover in this report for jest "

print(f"Query: {query}")

response = llm_tree_search(query, page_index, llm)
print(f"Response: {response}")

Query: what is unit test case cover in this report for jest 
Compressed Tree: [{'node_id': '0000', 'title': 'Jest Unit Test Coverage Report', 'summary': '# Jest Unit Test Coverage Report\n\ntnn-english-next | Times Now News (English)\n\nReport date: 2026-07-14 | Jest coverage + __tests__ inventory\n', 'page': 1, 'children': [{'node_id': '0001', 'title': '1. Executive Summary', 'summary': 'This report provides an overview of the TNN English Next.js Jest unit test suite, detailing its 324 test files, 3,277 static test cases, and 203 URL rewrite rules. It presents comprehensive coverage metrics (89.3% line coverage) and outlines the technical testing environment, which utilizes jest-environment-jsdom, Testing Library, and specific mocks for Next.js components.', 'page': 1}, {'node_id': '0002', 'title': '2. Coverage by Source Area', 'summary': 'This text presents a summary of code coverage metrics across various project areas—including components, pages, utilities, helpers, constants, hook